| | |
:----------------|---|
| **Nombre** | Santiago Escutia Ríos |
| **Fecha** | 3/5/2026 |
| **Expediente** | 757839 |

# EJERCICIOS CONCEPTUALES

## Ejercicio 1: Minimización de la Varianza

La idea es demostrar que el $\alpha$ de la ecuación (5.6) es el que minimiza:
$$f(\alpha) = \text{Var}(\alpha X + (1 - \alpha) Y)$$

**1. Expandir la varianza** usando sus propiedades:
$$f(\alpha) = \alpha^2 \sigma_X^2 + (1 - \alpha)^2 \sigma_Y^2 + 2\alpha(1 - \alpha) \sigma_{XY}$$

**2. Derivar e igualar a cero** para encontrar el mínimo:
$$f'(\alpha) = 2\alpha \sigma_X^2 - 2(1 - \alpha) \sigma_Y^2 + 2(1 - 2\alpha) \sigma_{XY} = 0$$

**3. Simplificar** dividiendo entre 2 y expandiendo:
$$0 = \alpha \sigma_X^2 - \sigma_Y^2 + \alpha \sigma_Y^2 + \sigma_{XY} - 2\alpha \sigma_{XY}$$

**4. Despejar $\alpha$:**
$$\alpha (\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}) = \sigma_Y^2 - \sigma_{XY}$$
$$\boxed{\alpha = \frac{\sigma_Y^2 - \sigma_{XY}}{\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}}}$$

Que es exactamente la ecuacion (5.6). Para confirmar que es minimo y no maximo, la segunda derivada $f''(\alpha) = 2(\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY})$ es positiva siempre que $X$ e $Y$ no esten perfectamente correlacionadas. ✓

## Ejercicio 2: Probabilidades en Bootstrap

Tenemos $n$ observaciones y muestreamos con reemplazo.

**(a)** Cada observacion tiene probabilidad $1/n$ de ser elegida, entonces la probabilidad de que la primera extraccion NO sea la $j$-esima es $1 - 1/n$.

**(b)** Como muestreamos con reemplazo, la segunda extraccion es independiente de la primera. La probabilidad es la misma: $1 - 1/n$.

**(c)** Para que $j$ no aparezca en ninguna de las $n$ extracciones, cada una debe fallar de forma independiente:
$$(1 - 1/n)^n$$

**(d)** Para $n = 5$:
$$P(j \in \text{muestra}) = 1 - (0.8)^5 = 1 - 0.32768 \approx 0.672$$

**(e)** Para $n = 100$:
$$P(j \in \text{muestra}) = 1 - (0.99)^{100} \approx 0.634$$

**(f)** Para $n = 10{,}000$:
$$P(j \in \text{muestra}) \approx 0.6321$$

**(g)** Conforme $n \to \infty$, sabemos que $(1 - 1/n)^n \to e^{-1}$, entonces la probabilidad de inclusion converge a $1 - 1/e \approx 0.6321$. Es interesante que ya con $n = 100$ estamos muy cerca del limite.

**(h)** La simulacion con 10,000 iteraciones deberia dar un promedio muy cercano a 0.632 por la ley de grandes numeros. La variabilidad entre corridas sera pequena a ese tamano de muestra.

## Ejercicio 3: Validacion Cruzada k-fold

**(a) Implementacion:**
1. Dividir el dataset en $k$ grupos de tamaño aproximadamente igual.
2. Entrenar el modelo $k$ veces: en cada iteracion, un grupo diferente sirve como conjunto de prueba y los $k-1$ restantes para entrenamiento.
3. El error final es el promedio de los $k$ errores.

**(b) Comparativas:**

- **vs. Validation Set:** el metodo de conjunto de validacion depende mucho de como queda la particion aleatoria inicial —puede dar errores muy variables. k-fold usa todos los datos en algun momento y da estimaciones mas estables.

- **vs. LOOCV:** LOOCV es el caso $k = n$. Tiene bajo sesgo pero puede tener varianza alta (porque los conjuntos de entrenamiento son casi identicos entre si) y es extremadamente caro computacionalmente. Con $k = 5$ o $k = 10$ se logra un mejor balance y es mucho mas practico.

## Ejercicio 4: Error Estandar de una Prediccion con Bootstrap

Para estimar la variabilidad de una prediccion $\hat{Y}$ en un punto $x_0$:

1. Generar $B$ muestras bootstrap (con reemplazo) del dataset original.
2. Ajustar el modelo en cada muestra $b$ y calcular $\hat{y}^{*b} = \hat{f}^{*b}(x_0)$.
3. El error estandar es la desviacion estandar de esas predicciones:
$$\text{SE}_B(\hat{y}) = \sqrt{\frac{1}{B-1} \sum_{b=1}^{B} \left(\hat{y}^{*b} - \bar{y}^*\right)^2}$$

La ventaja principal del Bootstrap es que no requiere asumir nada sobre la distribucion subyacente: funciona para cualquier estadistico, no solo para la media.

---
# EJERCICIOS PRACTICOS

## Ejercicio 5: Validation Set Approach en Default

Estimamos el error de prueba de una regresion logistica usando el metodo de conjunto de validacion.

### (a) Ajuste del modelo con todos los datos

In [10]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('Default.csv')
df['default_bin'] = df['default'].map({'No': 0, 'Yes': 1})

X = sm.add_constant(df[['income', 'balance']])
y = df['default_bin']

model_a = sm.Logit(y, X).fit()
print(model_a.summary())

Optimization terminated successfully.
         Current function value: 0.078948
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:            default_bin   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9997
Method:                           MLE   Df Model:                            2
Date:                Mon, 04 May 2026   Pseudo R-squ.:                  0.4594
Time:                        15:51:51   Log-Likelihood:                -789.48
converged:                       True   LL-Null:                       -1460.3
Covariance Type:            nonrobust   LLR p-value:                4.541e-292
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.000     -12.393     -10.688
income      2.081e-05   4.99

### (b) Estimacion del error de prueba

Pasos:
1. Dividir en entrenamiento y validacion (80/20).
2. Ajustar el modelo solo con entrenamiento.
3. Predecir en validacion: clasificar como 1 si $P > 0.5$.
4. Calcular la fraccion de clasificaciones incorrectas: $\text{Error} = \frac{1}{n_v}\sum I(y_i \neq \hat{y}_i)$

In [11]:
from sklearn.model_selection import train_test_split

train, validation = train_test_split(df, test_size=0.2, random_state=1)

X_train = sm.add_constant(train[['income', 'balance']])
model_b = sm.Logit(train['default_bin'], X_train).fit(disp=0)

X_val = sm.add_constant(validation[['income', 'balance']])
prob_val = model_b.predict(X_val)
pred_val = (prob_val > 0.5).astype(int)

error_b = (pred_val != validation['default_bin']).mean()
print(f"Tasa de error de validacion: {error_b:.4f}")

Tasa de error de validacion: 0.0270


### (c) Repeticion con distintas semillas

Repitiendo el proceso para ver cuanta variabilidad hay dependiendo de como caiga la particion:

In [12]:
def validation_error(seed):
    train, val = train_test_split(df, test_size=0.2, random_state=seed)
    X_t = sm.add_constant(train[['income', 'balance']])
    m = sm.Logit(train['default_bin'], X_t).fit(disp=0)
    X_v = sm.add_constant(val[['income', 'balance']])
    preds = (m.predict(X_v) > 0.5).astype(int)
    return (preds != val['default_bin']).mean()

for s in [10, 20, 30]:
    print(f"  Semilla {s}: {validation_error(s):.4f}")

  Semilla 10: 0.0260
  Semilla 20: 0.0230
  Semilla 30: 0.0285


Hay cierta variabilidad en el error segun la semilla, lo cual es una limitacion conocida del validation set approach.

### (d) Incluyendo la variable `student`

In [13]:
df['student_bin'] = df['student'].map({'No': 0, 'Yes': 1})

train, val = train_test_split(df, test_size=0.2, random_state=1)

X_train_d = sm.add_constant(train[['income', 'balance', 'student_bin']])
model_d = sm.Logit(train['default_bin'], X_train_d).fit(disp=0)

X_val_d = sm.add_constant(val[['income', 'balance', 'student_bin']])
preds_d = (model_d.predict(X_val_d) > 0.5).astype(int)

error_d = (preds_d != val['default_bin']).mean()
print(f"Sin student:     {error_b:.4f}")
print(f"Con student:     {error_d:.4f}")

Sin student:     0.0270
Con student:     0.0260


Si la diferencia es pequena o nula, concluimos que incluir `student` no aporta informacion adicional relevante cuando ya estan `income` y `balance` en el modelo.

---
## Ejercicio 6: Bootstrap para Errores Estandar

Comparamos los errores estandar de la regresion logistica obtenidos por teoria estadistica clasica vs. Bootstrap.

### (a) Errores estandar clasicos

In [14]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('Default.csv')
df['default_bin'] = df['default'].map({'No': 0, 'Yes': 1})

X = sm.add_constant(df[['income', 'balance']])
y = df['default_bin']

model_logit = sm.Logit(y, X).fit(disp=0)
print("Errores estandar (formula teorica):")
print(model_logit.bse.round(6))

Errores estandar (formula teorica):
const      0.434772
income     0.000005
balance    0.000227
dtype: float64


### (b) Funcion `boot_fn`

In [15]:
def boot_fn(data, index):
    X_b = sm.add_constant(data[['income', 'balance']].iloc[index])
    y_b = data['default_bin'].iloc[index]
    return sm.Logit(y_b, X_b).fit(disp=0).params

### (c) Bootstrap con $B = 1{,}000$ iteraciones

$$\text{SE}(\hat{\beta}) = \sqrt{\frac{1}{B-1} \sum_{b=1}^{B} \left(\hat{\beta}^{*b} - \bar{\beta}^*\right)^2}$$

In [16]:
import numpy as np

def run_bootstrap(data, B=1000):
    n = len(data)
    estimaciones = []
    for _ in range(B):
        idx = np.random.choice(n, n, replace=True)
        estimaciones.append(boot_fn(data, idx))
    return pd.DataFrame(estimaciones).std()

np.random.seed(42)
se_boot = run_bootstrap(df)
print("Errores estandar (Bootstrap, B=1000):")
print(se_boot.round(6))

Errores estandar (Bootstrap, B=1000):
const      0.434719
income     0.000005
balance    0.000232
dtype: float64


### (d) Comparacion y conclusion

Los dos metodos dan valores muy similares. Esto confirma que las formulas teoricas de la regresion logistica (basadas en la matriz de informacion de Fisher) son confiables para este dataset. El Bootstrap funciona como una validacion empirica sin asumir ninguna forma distribucional.

---
## Ejercicio 7: LOOCV con Regresion Logistica (Dataset Weekly)

### (a) Ajuste del modelo completo

(No encontr el dataset weekly lo habia hecho con default)

In [ ]:
import pandas as pd
import statsmodels.api as sm

df_w = pd.read_csv('Weekly.csv')
df_w['Direction_bin'] = df_w['Direction'].map({'Down': 0, 'Up': 1})

X = sm.add_constant(df_w[['Lag1', 'Lag2']])
y = df_w['Direction_bin']

model_full = sm.Logit(y, X).fit(disp=0)
print(model_full.summary())

### (b) Modelo sin la primera observacion

In [ ]:
X_minus_1 = X.drop(X.index[0])
y_minus_1 = y.drop(y.index[0])

model_minus_1 = sm.Logit(y_minus_1, X_minus_1).fit(disp=0)

### (c) Prediccion para la primera observacion

In [ ]:
prob_1 = model_minus_1.predict(X.iloc[0:1]).values[0]
pred_1 = 1 if prob_1 > 0.5 else 0

print(f"Probabilidad: {prob_1:.4f} | Prediccion: {pred_1} | Real: {y.iloc[0]}")
print(f"Correcto: {pred_1 == y.iloc[0]}")

### (d) Bucle LOOCV completo

In [ ]:
import numpy as np

n = len(df_w)
errores = []

for i in range(n):
    X_train = X.drop(X.index[i])
    y_train = y.drop(y.index[i])
    m_i = sm.Logit(y_train, X_train).fit(disp=0)
    
    prob_i = m_i.predict(X.iloc[i:i+1]).values[0]
    pred_i = 1 if prob_i > 0.5 else 0
    errores.append(int(pred_i != y.iloc[i]))

### (e) Error LOOCV

In [ ]:
loocv_error = np.mean(errores)
print(f"Error de prueba (LOOCV): {loocv_error:.4f}  ({loocv_error:.2%})")

Un error bastante alto, lo cual tiene sentido: predecir la direccion del mercado usando solo dos rezagos es una tarea muy dificil. Aun asi, LOOCV nos da una estimacion confiable de ese error real sin depender de ninguna particion aleatoria.

---
## Ejercicio 8: CV en Datos Simulados

### (a) Generacion de datos

El modelo generativo es $Y = X - 2X^2 + \epsilon$ con $\epsilon \sim N(0,1)$.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
x = rng.normal(size=100)
y = x - 2 * x**2 + rng.normal(size=100)

# n = 100, p = 2 en el modelo verdadero (X y X^2)
print(f"n = {len(x)}")
print(f"Modelo generativo: Y = X - 2X^2 + epsilon")

### (b) Diagrama de dispersion

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.scatter(x, y, alpha=0.6, edgecolors='gray', linewidths=0.4)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Datos simulados: Y = X - 2X^2 + epsilon")
plt.tight_layout()
plt.show()

Se ve claramente la forma de parabola invertida, dominada por el termino $-2X^2$.

### (c) Error LOOCV para polinomios de grado 1 a 4

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.preprocessing import PolynomialFeatures

df_sim = pd.DataFrame({'X': x, 'Y': y})
loo = LeaveOneOut()
lr = LinearRegression()

print("Grado | Error LOOCV (MSE)")
print("-" * 25)
for grado in range(1, 5):
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    X_poly = poly.fit_transform(df_sim[['X']])
    scores = cross_val_score(lr, X_poly, df_sim['Y'],
                             scoring='neg_mean_squared_error', cv=loo)
    mse = np.mean(np.abs(scores))
    print(f"  {grado}   | {mse:.4f}")

### (d) Cambio de semilla

Si se usa otra semilla, los datos cambian y los valores absolutos de MSE seran diferentes, pero el orden entre modelos deberia ser el mismo. Esto es porque LOOCV en si mismo no tiene componente aleatorio: para un conjunto de datos fijo, siempre hace las mismas particiones.

### (e) Modelo optimo

El menor error corresponde al polinomio de **grado 2**. Era de esperarse porque ese es exactamente el modelo que genero los datos. Agregar terminos de grado 3 o 4 aumenta la complejidad sin mejorar la prediccion real, y puede introducir algo de sobreajuste.

### (f) Significancia estadistica de los coeficientes

In [ ]:
import statsmodels.formula.api as smf

model_4 = smf.ols('Y ~ X + I(X**2) + I(X**3) + I(X**4)', data=df_sim).fit()
print(model_4.summary().tables[1])

Los terminos $X$ y $X^2$ son significativos ($p < 0.05$) mientras que $X^3$ y $X^4$ no lo son. Coincide perfectamente con lo que muestra la validacion cruzada: ambos metodos senalan al modelo cuadratico como el correcto.

---
## Ejercicio 9: Bootstrap en el Dataset Boston

### (a) Media estimada

In [17]:
import pandas as pd
import numpy as np

df_boston = pd.read_excel('Boston Housing Dataset 1978.xlsx')

mu_hat = df_boston['MEDV'].mean()
print(f"Media estimada (mu_hat): {mu_hat:.4f}")

Media estimada (mu_hat): 23.7504


### (b) Error estandar por formula

In [18]:
n = len(df_boston)
se_formula = df_boston['MEDV'].std() / np.sqrt(n)
print(f"Error estandar (formula): {se_formula:.4f}")

Error estandar (formula): 0.3916


### (c) Error estandar por Bootstrap

In [19]:
np.random.seed(0)
B = 1000

boot_means = [
    df_boston['MEDV'].sample(n=n, replace=True).mean()
    for _ in range(B)
]

se_boot = np.std(boot_means)
print(f"Error estandar (Bootstrap): {se_boot:.4f}")
print(f"Error estandar (formula):   {se_formula:.4f}")

Error estandar (Bootstrap): 0.4151
Error estandar (formula):   0.3916


### (d) Intervalo de confianza al 95%

In [20]:
ci = [mu_hat - 2*se_boot, mu_hat + 2*se_boot]
print(f"IC 95%: [{ci[0]:.4f}, {ci[1]:.4f}]")

IC 95%: [22.9203, 24.5806]


### (e) Mediana estimada

In [21]:
med_hat = df_boston['MEDV'].median()
print(f"Mediana estimada: {med_hat:.4f}")

Mediana estimada: 21.9500


### (f) Error estandar de la mediana por Bootstrap

A diferencia de la media, no hay formula cerrada para el error estandar de la mediana, por eso Bootstrap es especialmente util aqui.

In [22]:
boot_medians = [
    df_boston['MEDV'].sample(n=n, replace=True).median()
    for _ in range(B)
]

se_med = np.std(boot_medians)
print(f"Error estandar de la mediana: {se_med:.4f}")

Error estandar de la mediana: 0.3065


### (g) Percentil 10

In [23]:
pct10_hat = df_boston['MEDV'].quantile(0.1)
print(f"Percentil 10 estimado: {pct10_hat:.4f}")

Percentil 10 estimado: 14.5000


### (h) Error estandar del percentil 10

In [24]:
boot_pct10 = [
    df_boston['MEDV'].sample(n=n, replace=True).quantile(0.1)
    for _ in range(B)
]

se_pct10 = np.std(boot_pct10)
print(f"Error estandar del percentil 10: {se_pct10:.4f}")

Error estandar del percentil 10: 0.3914


**Conclusion:** El Bootstrap es consistente con la formula teorica para la media, y nos permite extender el analisis a estadisticos como la mediana o percentiles para los cuales no existe una formula simple. Es la misma logica en todos los casos: muestrear con reemplazo, calcular el estadistico, repetir muchas veces y medir la dispersion de los resultados.